<a href="https://colab.research.google.com/github/meedooatef22/arabic-dialect-classification/blob/main/notebooks/01_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Mount Google Drive and Create Project Directories

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive to Google Colab
drive.mount('/content/drive')

# 2. Create the main project directory and data folders inside Google Drive
project_path = '/content/drive/MyDrive/arabic-dialect-classification'
os.makedirs(f'{project_path}/data/raw', exist_ok=True)
os.makedirs(f'{project_path}/data/processed', exist_ok=True)

print("Drive mounted and directories created successfully! 🚀")

Mounted at /content/drive
Drive mounted and directories created successfully! 🚀


### Step 2: Download and Save the MADAR Dataset

In [ ]:
import kagglehub
import shutil

print("Downloading the MADAR dataset from Kaggle...")
path = kagglehub.dataset_download("bechirbentekfa/madar-dataset")
print("Downloaded to temporary path:", path)

raw_data_path = '/content/drive/MyDrive/arabic-dialect-classification/data/raw'

# Recursively copy all files and folders
shutil.copytree(path, raw_data_path, dirs_exist_ok=True)

print(f"Dataset files successfully copied to: {raw_data_path} 🚀")

# Display all files saved in Drive
print("Files now available in Drive:")
for root, dirs, files in os.walk(raw_data_path):
    for file in files:
        print(file)

Using Colab cache for faster access to the 'madar-dataset' dataset.
Downloaded to temporary path: /kaggle/input/madar-dataset
Dataset files successfully copied to: /content/drive/MyDrive/arabic-dialect-classification/data/raw 🚀
Files now available in Drive:
README.txt
LICENSE.txt
The_MADAR_Arabic_Dialect_Corpus_and_Lexicon.pdf
Icon_
MADAR.corpus.Jerusalem.tsv
MADAR.corpus.Tunis.tsv
MADAR.corpus.Mosul.tsv
MADAR.corpus.Sfax.tsv
MADAR.corpus.Rabat.tsv
MADAR.corpus.Basra.tsv
MADAR.corpus.Muscat.tsv
MADAR.corpus.Khartoum.tsv
MADAR.corpus.Amman.tsv
MADAR.corpus.English.index.tsv
MADAR.corpus.Sanaa.tsv
MADAR.corpus.Baghdad.tsv
MADAR.corpus.Jeddah.tsv
MADAR.corpus.Cairo.tsv
MADAR.corpus.Algiers.tsv
MADAR.corpus.Alexandria.tsv
MADAR.corpus.Aleppo.tsv
MADAR.corpus.Fes.tsv
MADAR.corpus.Beirut.tsv
MADAR.corpus.Tripoli.tsv
MADAR.corpus.Doha.tsv
MADAR.corpus.MSA.tsv
MADAR.corpus.Riyadh.tsv
MADAR.corpus.Salt.tsv
Icon_
MADAR.corpus.Damascus.tsv
MADAR.corpus.Aswan.tsv
MADAR.corpus.Benghazi.tsv
MADAR.co

### Step 3.1: Select and Combine 5 Main Dialects

In [ ]:
import pandas as pd

raw_data_path = '/content/drive/MyDrive/arabic-dialect-classification/data/raw'
selected_dialects = ['Cairo', 'Riyadh', 'Beirut', 'Tunis', 'MSA']
dataframes = []

print("Loading selected dialects...")

# Loop through files and load only the selected dialects
for root, dirs, files in os.walk(raw_data_path):
    for file in files:
        for dialect in selected_dialects:
            if f"MADAR.corpus.{dialect}.tsv" in file:
                file_path = os.path.join(root, file)
                # Load the TSV file
                df = pd.read_csv(file_path, sep='\t', header=None, names=['text'])
                df['label'] = dialect
                dataframes.append(df)

# Combine them into one table
full_df = pd.concat(dataframes, ignore_index=True)

print(f"Total rows loaded: {len(full_df)}")
display(full_df.head())

Loading selected dialects...
Total rows loaded: 100010


,text,label
0,sent,Tunis
1,موجود في اخر الكولوار. و نجيبهولك توا. كان تست...,Tunis
2,تعملوا تبديلات؟,Tunis
3,نحبوا طاولة قريبة مالشباك.,Tunis
4,اهوكا غادي، بالضبط قدام البيرو متاع الارشادات ...,Tunis


### Step 3.2: Clean and Normalize Arabic Text

In [ ]:
# Install pyarabic library for handling Arabic text
!pip install pyarabic -q

import re
import pyarabic.araby as araby

# 1. Drop missing values and duplicates to save time
full_df.dropna(inplace=True)
full_df.drop_duplicates(subset=['text'], inplace=True)
print(f"Rows after removing duplicates: {len(full_df)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 1.9 MB/s eta 0:00:00
Rows after removing duplicates: 49414


### Step 3.3: Apply Arabic Text Cleaning Function

In [ ]:
import re
import pyarabic.araby as araby

# Define the text cleaning function
def clean_arabic_text(text):
    text = str(text)

    # Strip diacritics (Tashkeel) and Tatweel (elongation)
    text = araby.strip_tashkeel(text)
    text = araby.strip_tatweel(text)

    # Normalize Alef, Teh Marboota, and Alef Maksura
    text = re.sub(r'[إأآ]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ى', 'ي', text)

    # Remove non-Arabic characters, numbers, and punctuation
    text = re.sub(r'[^\w\s\u0621-\u064A]', ' ', text)
    text = re.sub(r'[0-9a-zA-Z]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

print("Applying cleaning function... (this might take a few seconds)")

# Apply the function to create a new column 'cleaned_text'
full_df['cleaned_text'] = full_df['text'].apply(clean_arabic_text)

# Display the first 5 rows to see the difference
display(full_df[['text', 'cleaned_text', 'label']].head())

Applying cleaning function... (this might take a few seconds)


,text,cleaned_text,label
0,sent,,Tunis
1,موجود في اخر الكولوار. و نجيبهولك توا. كان تست...,موجود في اخر الكولوار و نجيبهولك توا كان تستحق...,Tunis
2,تعملوا تبديلات؟,تعملوا تبديلات,Tunis
3,نحبوا طاولة قريبة مالشباك.,نحبوا طاوله قريبه مالشباك,Tunis
4,اهوكا غادي، بالضبط قدام البيرو متاع الارشادات ...,اهوكا غادي بالضبط قدام البيرو متاع الارشادات ا...,Tunis


### Step 3.4: Save the Cleaned Dataset for the Next Phase

In [ ]:
processed_data_path = '/content/drive/MyDrive/arabic-dialect-classification/data/processed'
output_file = os.path.join(processed_data_path, 'cleaned_dataset.csv')

print("Saving the cleaned dataset...")
# Save the dataframe to a CSV file without the index
full_df.to_csv(output_file, index=False)

print(f"Dataset saved successfully to: {output_file} 🚀")

Saving the cleaned dataset...
Dataset saved successfully to: /content/drive/MyDrive/arabic-dialect-classification/data/processed/cleaned_dataset.csv 🚀
